
# Bengali NLP Safety V27 FIXED — 0.600+ Attempt

**Protected Kaggle champion: V14 = 0.586**

V27 is designed around the strongest evidence from Bengali abusive-language NLP:

- `saroarj/BanglaHateBert` — Bengali abuse-domain adapted encoder
- `Hate-speech-CNERG/bengali-abusive-MuRIL` — Bengali abusive MuRIL
- character TF-IDF logistic regression — robust to spelling variation, censorship, Banglish and Romanization
- layer-wise learning-rate decay (LLRD)
- light FGM adversarial training
- grouped 5-fold train-only OOF
- nested/cross-fitted threshold selection
- official 500-row validation is kept untouched until the final gate

## Submission policy

**Do not submit V27 automatically.**

At the end, the notebook compares V27 against reconstructed V14 on the untouched validation set.

A V27 hybrid is marked `SUBMIT_CANDIDATE=True` only if it beats V14 validation by a meaningful margin.

Run all cells, download `v27_results.zip`, and send it to ChatGPT first.


**FIX:** AMP/FGM double-`unscale_()` crash removed.


In [ ]:

# 1. Upload V27 data
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v27_data.zip" in uploaded, "Upload the supplied v27_data.zip."

DATA_DIR = pathlib.Path("/content/v27_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v27_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Dependencies and model preflight
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
    "tqdm",
    "safetensors",
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/csebuetnlp/normalizer"
])

from transformers import AutoConfig, AutoTokenizer
from normalizer import normalize

MODEL_NAMES = {
    "banglahate": "saroarj/BanglaHateBert",
    "abusive_muril": "Hate-speech-CNERG/bengali-abusive-MuRIL",
}

for key, name in MODEL_NAMES.items():
    print("Checking:", key, name)
    _ = AutoConfig.from_pretrained(name)
    _ = AutoTokenizer.from_pretrained(name)

print("✅ Model preflight passed.")


In [ ]:

# 3. Imports, data and settings
import gc, re, math, json, random, unicodedata, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from normalizer import normalize

SEED = 2026
N_FOLDS = 5
EPOCHS = 3
SNAPSHOT_EPOCHS = [2,3]
MAX_LENGTH = 128
WEIGHT_DECAY = 0.01
LAYER_DECAY = 0.90
WARMUP_RATIO = 0.10

# Light FGM: only epochs 2-3 and every other batch.
USE_FGM = True
FGM_EPS = 0.35

LABELS = [
    "is_toxic",
    "implicit_toxicity",
    "sarcasm",
    "contains_profanity",
    "threatening",
    "identity_attack",
    "sexual_content",
]

MODEL_NAMES = {
    "banglahate": "saroarj/BanglaHateBert",
    "abusive_muril": "Hate-speech-CNERG/bengali-abusive-MuRIL",
}

MODEL_CFG = {
    "banglahate": {
        "batch": 8,
        "accum": 2,
        "backbone_lr": 1.5e-5,
        "head_lr": 3.0e-5,
        "text_col": "bangla_text",
    },
    "abusive_muril": {
        "batch": 4,
        "accum": 4,
        "backbone_lr": 1.0e-5,
        "head_lr": 2.0e-5,
        "text_col": "raw_text",
    },
}

RUN_DIR = Path("/content/v27_output")
RUN_DIR.mkdir(exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
assert torch.cuda.is_available(), "Enable a T4 GPU."

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
champ = pd.read_csv(DATA_DIR/"submission_v14_champion.csv", dtype={"ID":str})
v2v = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2t = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})

with np.load(DATA_DIR/"old_banglabert_predictions.npz") as z:
    old_bang_v = z["valid"].copy()
    old_bang_t = z["test"].copy()

with np.load(DATA_DIR/"old_xlmr_predictions.npz") as z:
    old_xl_v = z["valid"].copy()
    old_xl_t = z["test"].copy()

assert len(valid)==500 and len(test)==1000
assert champ["ID"].tolist()==test["ID"].tolist()
assert v2v["ID"].tolist()==valid["ID"].tolist()
assert v2t["ID"].tolist()==test["ID"].tolist()

print("train / validation / test:",len(train),len(valid),len(test))
print("label prevalence:")
display(train[LABELS].mean().to_frame("train_rate").round(4))


In [ ]:

# 4. Bengali-aware cleaning and grouped folds
BN_RE = re.compile(r"[\u0980-\u09FF]")
LAT_RE = re.compile(r"[A-Za-z]")

def basic_clean(x):
    x = unicodedata.normalize("NFKC",str(x))
    x = x.replace("\\u200c"," ").replace("\\u200d"," ")
    return re.sub(r"\\s+"," ",x).strip()

def bangla_clean(x):
    x = basic_clean(x)
    try:
        x = normalize(x)
    except Exception:
        pass
    return re.sub(r"\\s+"," ",x).strip()

def group_key(x):
    return basic_clean(x).lower()

for df in [train,valid,test]:
    df["raw_text"] = df["text"].map(basic_clean)
    df["bangla_text"] = df["text"].map(bangla_clean)
    df["group_key"] = df["text"].map(group_key)

# Greedy multilabel grouped split.
groups = train.groupby("group_key",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

global_pos = np.array([train[l].sum() for l in LABELS],float)
total = float(len(train))
rarity = 1/np.sqrt(np.maximum(global_pos,1))
mat = groups[[f"sum_{l}" for l in LABELS]].to_numpy(float)

rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+0.005*groups["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(groups))*1e-8))

fold_pos=np.zeros((N_FOLDS,len(LABELS)))
fold_size=np.zeros(N_FOLDS)
assignment={}

for idx in order:
    row=groups.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in LABELS],float)
    size=float(row["size"])

    costs=[]
    for f in range(N_FOLDS):
        p=fold_pos.copy()
        s=fold_size.copy()
        p[f]+=vec
        s[f]+=size

        cost=(
            np.mean((p/np.maximum(global_pos[None,:],1)-1/N_FOLDS)**2)
            + 0.30*np.mean((s/total-1/N_FOLDS)**2)
        )
        costs.append(cost)

    best=int(np.argmin(costs))
    assignment[row["group_key"]]=best
    fold_pos[best]+=vec
    fold_size[best]+=size

train["fold"]=train["group_key"].map(assignment).astype(int)

assert train.groupby("group_key")["fold"].nunique().max()==1
assert set(train["fold"])==set(range(N_FOLDS))

print("fold sizes:",train.groupby("fold").size().to_dict())
display(train.groupby("fold")[LABELS].sum())


In [ ]:

# 5. Dataset, ASL loss, FGM and LLRD helpers
class TextDataset(Dataset):
    def __init__(self,frame,tokenizer,text_col,with_labels=True):
        self.texts=frame[text_col].astype(str).tolist()
        self.y=frame[LABELS].to_numpy(np.float32) if with_labels else None
        self.tok=tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,i):
        enc=self.tok(
            self.texts[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.y is not None:
            enc["labels"]=self.y[i]
        return enc

class Collator:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )

        batch=self.tok.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )
        if y is not None:
            batch["labels"]=y
        return batch

# Competition-shaped asymmetric multi-label loss.
METRIC_W = torch.tensor(
    [0.5]+[0.5/6]*6,
    dtype=torch.float32
)

def asymmetric_loss(logits,y,gamma_neg=2.0,gamma_pos=0.0,clip=0.05):
    x=torch.sigmoid(logits.float())
    xs_pos=x
    xs_neg=1-x

    if clip and clip>0:
        xs_neg=(xs_neg+clip).clamp(max=1)

    loss=(
        y*torch.log(xs_pos.clamp_min(1e-8))
        +(1-y)*torch.log(xs_neg.clamp_min(1e-8))
    )

    if gamma_neg>0 or gamma_pos>0:
        pt=xs_pos*y + xs_neg*(1-y)
        gamma=gamma_pos*y + gamma_neg*(1-y)
        loss *= torch.pow(1-pt,gamma)

    per_label=-loss
    return (
        per_label
        * METRIC_W.to(logits.device).unsqueeze(0)
    ).sum(dim=1).mean()

class FGM:
    def __init__(self,model,eps=0.35):
        self.model=model
        self.eps=eps
        self.backup={}

    def attack(self):
        for n,p in self.model.named_parameters():
            if (
                p.requires_grad
                and p.grad is not None
                and ("word_embeddings" in n or "embeddings.word" in n)
            ):
                norm=torch.norm(p.grad)
                if torch.isfinite(norm) and norm>0:
                    self.backup[n]=p.data.clone()
                    p.data.add_(self.eps*p.grad/norm)

    def restore(self):
        for n,p in self.model.named_parameters():
            if n in self.backup:
                p.data=self.backup[n]
        self.backup={}

def make_llrd_groups(model,base_lr,head_lr,layer_decay=LAYER_DECAY):
    n_layers=int(getattr(model.config,"num_hidden_layers",12))
    groups={}

    def get_lr(name):
        if "classifier" in name or "score" in name:
            return head_lr

        m=re.search(r"encoder\\.layer\\.(\\d+)",name)
        if m:
            layer=int(m.group(1))
            return base_lr*(layer_decay**(n_layers-1-layer))

        return base_lr*(layer_decay**n_layers)

    for name,p in model.named_parameters():
        if not p.requires_grad:
            continue

        lr=get_lr(name)
        no_decay=(
            name.endswith("bias")
            or "LayerNorm.weight" in name
            or "layer_norm.weight" in name
        )
        wd=0.0 if no_decay else WEIGHT_DECAY
        key=(lr,wd)
        groups.setdefault(key,[]).append(p)

    return [
        {"params":params,"lr":lr,"weight_decay":wd}
        for (lr,wd),params in groups.items()
    ]

@torch.no_grad()
def predict(model,loader):
    model.eval()
    parts=[]
    for b in loader:
        b.pop("labels",None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}
        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp).logits
        parts.append(torch.sigmoid(logits.float()).cpu().numpy())

    out=np.concatenate(parts)
    assert np.isfinite(out).all()
    return out


In [ ]:

# 6. Train one transformer expert with grouped 5-fold OOF
fold_seeds=[2026,17,3407,1337,42]

def train_transformer_expert(expert):
    model_name=MODEL_NAMES[expert]
    cfg=MODEL_CFG[expert]
    text_col=cfg["text_col"]

    oof=np.zeros((len(train),len(LABELS)),np.float32)
    valid_fold=[]
    test_fold=[]
    history=[]

    tokenizer=AutoTokenizer.from_pretrained(model_name)

    for fold in range(N_FOLDS):
        tr=train[train.fold!=fold].copy()
        va=train[train.fold==fold].copy()

        assert set(tr.group_key).isdisjoint(set(va.group_key))

        cache=RUN_DIR/f"{expert}_fold_{fold}.npz"
        hp=RUN_DIR/f"{expert}_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["va_index"]
                vp=z["valid"]
                ev=z["external_valid"]
                tp=z["test"]
            if (
                np.array_equal(idx,va.index.to_numpy())
                and vp.shape==(len(va),len(LABELS))
            ):
                print("✅ reuse",expert,"fold",fold)
                oof[idx]=vp
                valid_fold.append(ev)
                test_fold.append(tp)
                if hp.exists():
                    history.extend(pd.read_csv(hp).to_dict("records"))
                continue

        print("\\n"+"="*72)
        print(expert,"fold",fold,"train",len(tr),"valid",len(va))

        seed_everything(fold_seeds[fold])
        gc.collect()
        torch.cuda.empty_cache()

        col=Collator(tokenizer)

        tl=DataLoader(
            TextDataset(tr,tokenizer,text_col,True),
            batch_size=cfg["batch"],
            shuffle=True,
            num_workers=0,
            pin_memory=True,
            collate_fn=col,
        )
        vl=DataLoader(
            TextDataset(va,tokenizer,text_col,True),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=col,
        )
        evl=DataLoader(
            TextDataset(valid,tokenizer,text_col,False),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=col,
        )
        tel=DataLoader(
            TextDataset(test,tokenizer,text_col,False),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=col,
        )

        model=AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(LABELS),
            problem_type="multi_label_classification",
            ignore_mismatched_sizes=True,
        ).to("cuda")

        groups=make_llrd_groups(
            model,
            cfg["backbone_lr"],
            cfg["head_lr"],
        )
        optimizer=torch.optim.AdamW(groups)

        total_steps=math.ceil(len(tl)/cfg["accum"])*EPOCHS
        scheduler=get_linear_schedule_with_warmup(
            optimizer,
            int(total_steps*WARMUP_RATIO),
            total_steps
        )
        scaler=torch.amp.GradScaler("cuda")
        fgm=FGM(model,FGM_EPS)

        snap_va=[]
        snap_ev=[]
        snap_te=[]
        fold_hist=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            run=0.0
            seen=0

            for step,b in enumerate(
                tqdm(tl,desc=f"{expert} f{fold} e{epoch}/{EPOCHS}")
            ):
                y=b.pop("labels").to("cuda",non_blocking=True)
                inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inp).logits

                loss=asymmetric_loss(logits,y)

                if not torch.isfinite(loss):
                    raise FloatingPointError("non-finite loss")

                scaler.scale(loss/cfg["accum"]).backward()

                # Light adversarial pass only after epoch 1 and every other batch.
                if USE_FGM and epoch>=2 and step%2==0:
                    # IMPORTANT: do NOT call scaler.unscale_(optimizer) here.
                    # FGM normalizes the gradient direction, so the common AMP
                    # scale factor cancels out. We unscale exactly once later,
                    # immediately before clipping/optimizer.step().
                    fgm.attack()

                    with torch.autocast(device_type="cuda",dtype=torch.float16):
                        adv_logits=model(**inp).logits
                    adv_loss=asymmetric_loss(adv_logits,y)

                    fgm.restore()
                    scaler.scale(adv_loss/cfg["accum"]).backward()

                run+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%cfg["accum"]==0 or step+1==len(tl):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

                    old=scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()

                    if scaler.get_scale()>=old:
                        scheduler.step()

                    optimizer.zero_grad(set_to_none=True)

            rec={
                "expert":expert,
                "fold":fold,
                "epoch":epoch,
                "loss":run/max(seen,1),
            }
            fold_hist.append(rec)
            history.append(rec)
            pd.DataFrame(fold_hist).to_csv(hp,index=False)
            print(rec)

            if epoch in SNAPSHOT_EPOCHS:
                snap_va.append(predict(model,vl))
                snap_ev.append(predict(model,evl))
                snap_te.append(predict(model,tel))

        vp=np.mean(snap_va,axis=0)
        ev=np.mean(snap_ev,axis=0)
        tp=np.mean(snap_te,axis=0)

        oof[va.index.to_numpy()]=vp
        valid_fold.append(ev)
        test_fold.append(tp)

        np.savez_compressed(
            cache,
            va_index=va.index.to_numpy(),
            valid=vp,
            external_valid=ev,
            test=tp,
        )

        del model,optimizer,scheduler,scaler,fgm
        del tl,vl,evl,tel
        gc.collect()
        torch.cuda.empty_cache()

    external_valid=np.mean(valid_fold,axis=0)
    test_prob=np.mean(test_fold,axis=0)

    np.savez_compressed(
        RUN_DIR/f"{expert}_oof_valid_test.npz",
        oof=oof,
        validation=external_valid,
        test=test_prob,
    )
    pd.DataFrame(history).to_csv(
        RUN_DIR/f"{expert}_training_history.csv",
        index=False
    )

    return oof,external_valid,test_prob


In [ ]:

# 7. Train the two domain transformer experts
transformer_outputs={}

for expert in ["banglahate","abusive_muril"]:
    print("\\n"+"#"*80)
    print("TRAIN:",expert)
    print("#"*80)
    transformer_outputs[expert]=train_transformer_expert(expert)

bh_oof,bh_val,bh_test=transformer_outputs["banglahate"]
mu_oof,mu_val,mu_test=transformer_outputs["abusive_muril"]

print("✅ Transformer experts complete.")


In [ ]:

# 8. Character TF-IDF OOF expert
# Character n-grams capture Bengali spelling noise, censored profanity and Romanized/Banglish forms.

char_oof=np.zeros((len(train),len(LABELS)),np.float32)
char_val_folds=[]
char_test_folds=[]

for fold in range(N_FOLDS):
    tr=train[train.fold!=fold]
    va=train[train.fold==fold]

    vec=TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2,6),
        min_df=2,
        max_df=0.995,
        max_features=180000,
        sublinear_tf=True,
        dtype=np.float32,
    )

    Xtr=vec.fit_transform(tr["raw_text"])
    Xva=vec.transform(va["raw_text"])
    Xev=vec.transform(valid["raw_text"])
    Xte=vec.transform(test["raw_text"])

    fold_va=np.zeros((len(va),len(LABELS)),np.float32)
    fold_ev=np.zeros((len(valid),len(LABELS)),np.float32)
    fold_te=np.zeros((len(test),len(LABELS)),np.float32)

    for j,label in enumerate(LABELS):
        clf=LogisticRegression(
            C=3.0,
            solver="liblinear",
            max_iter=1000,
            random_state=SEED+fold,
        )
        clf.fit(Xtr,tr[label].to_numpy(int))

        fold_va[:,j]=clf.predict_proba(Xva)[:,1]
        fold_ev[:,j]=clf.predict_proba(Xev)[:,1]
        fold_te[:,j]=clf.predict_proba(Xte)[:,1]

    char_oof[va.index.to_numpy()]=fold_va
    char_val_folds.append(fold_ev)
    char_test_folds.append(fold_te)

    print("char fold",fold,"done | features",Xtr.shape[1])

char_val=np.mean(char_val_folds,axis=0)
char_test=np.mean(char_test_folds,axis=0)

np.savez_compressed(
    RUN_DIR/"char_tfidf_oof_valid_test.npz",
    oof=char_oof,
    validation=char_val,
    test=char_test,
)

print("✅ Character expert complete.")


In [ ]:

# 9. Fast official metrics and cross-fitted thresholding
y_train=train[LABELS].to_numpy(np.int8)
folds=train["fold"].to_numpy(int)

def f1_pos(y,pred):
    y=np.asarray(y,np.int8)
    pred=np.asarray(pred,np.int8)
    tp=np.sum((y==1)&(pred==1))
    fp=np.sum((y==0)&(pred==1))
    fn=np.sum((y==1)&(pred==0))
    den=2*tp+fp+fn
    return float(2*tp/den) if den else 0.0

def f1_macro_binary(y,pred):
    vals=[]
    y=np.asarray(y,np.int8)
    pred=np.asarray(pred,np.int8)
    for c in [0,1]:
        tp=np.sum((y==c)&(pred==c))
        fp=np.sum((y!=c)&(pred==c))
        fn=np.sum((y==c)&(pred!=c))
        den=2*tp+fp+fn
        vals.append(2*tp/den if den else 0.0)
    return float(np.mean(vals))

def metric(label,y,pred):
    return f1_macro_binary(y,pred) if label=="is_toxic" else f1_pos(y,pred)

def best_threshold(label,y,p):
    y=np.asarray(y,np.int8)
    p=np.asarray(p,float)

    vals=np.unique(p)
    if len(vals)>700:
        vals=np.quantile(vals,np.linspace(0,1,700))
    cand=np.unique(np.r_[0.0,vals,1.000001])

    order=np.argsort(p)
    ps=p[order]
    ys=y[order]

    idx=np.searchsorted(ps,cand,side="left")
    pp=np.r_[0,np.cumsum(ys)]
    pn=np.r_[0,np.cumsum(1-ys)]

    pos=pp[-1]
    neg=pn[-1]

    tp=pos-pp[idx]
    fp=neg-pn[idx]
    fn=pp[idx]
    tn=pn[idx]

    if label=="is_toxic":
        f1p=np.divide(
            2*tp,2*tp+fp+fn,
            out=np.zeros_like(tp,dtype=float),
            where=(2*tp+fp+fn)!=0
        )
        f1n=np.divide(
            2*tn,2*tn+fp+fn,
            out=np.zeros_like(tn,dtype=float),
            where=(2*tn+fp+fn)!=0
        )
        scores=(f1p+f1n)/2
    else:
        scores=np.divide(
            2*tp,2*tp+fp+fn,
            out=np.zeros_like(tp,dtype=float),
            where=(2*tp+fp+fn)!=0
        )

    best=scores.max()
    tied=cand[np.isclose(scores,best,atol=1e-12,rtol=0)]
    return float(np.median(tied))

def crossfit(label,y,p):
    pred=np.zeros(len(y),np.int8)
    fs=[]
    ts=[]

    for f in range(N_FOLDS):
        held=folds==f
        t=best_threshold(label,y[~held],p[~held])
        pred[held]=(p[held]>=t).astype(np.int8)
        ts.append(t)
        fs.append(metric(label,y[held],pred[held]))

    return {
        "score":metric(label,y,pred),
        "fold_scores":np.asarray(fs,float),
        "thresholds":np.asarray(ts,float),
        "prediction":pred,
    }

def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))


In [ ]:

# 10. Train-only ensemble search
# Coarse simplex only: enough diversity without hundreds of meta-parameters.

weights=[]
grid=[0.0,0.25,0.50,0.75,1.0]
for a in grid:
    for b in grid:
        c=1-a-b
        if c < -1e-9:
            continue
        if min(abs(c-x) for x in grid) > 1e-9:
            continue
        weights.append((a,b,round(c,2)))

rows=[]
best_by_label={}

for j,label in enumerate(LABELS):
    y=y_train[:,j]
    best=None

    for mode in ["prob","logit"]:
        for w_bh,w_mu,w_ch in weights:
            if mode=="prob":
                p=(
                    w_bh*bh_oof[:,j]
                    +w_mu*mu_oof[:,j]
                    +w_ch*char_oof[:,j]
                )
            else:
                p=sigmoid(
                    w_bh*logit(bh_oof[:,j])
                    +w_mu*logit(mu_oof[:,j])
                    +w_ch*logit(char_oof[:,j])
                )

            cv=crossfit(label,y,p)
            mean=float(cv["fold_scores"].mean())
            std=float(cv["fold_scores"].std())

            # Stability matters because the hidden leaderboard repeatedly punished
            # validation-specific gains.
            robust=mean-0.05*std

            rec={
                "label":label,
                "mode":mode,
                "w_banglahate":w_bh,
                "w_abusive_muril":w_mu,
                "w_char":w_ch,
                "crossfit_score":float(cv["score"]),
                "mean_fold":mean,
                "std_fold":std,
                "robust_score":robust,
                "median_threshold":float(np.median(cv["thresholds"])),
                "fold_scores":"|".join(f"{x:.6f}" for x in cv["fold_scores"]),
            }
            rows.append(rec)

            # Tie-break toward transformer diversity, not pure lexical model.
            key=(robust,cv["score"],-std,-w_ch)

            if best is None or key>best["key"]:
                best={
                    "key":key,
                    "rec":rec,
                    "cv":cv,
                    "oof_prob":p,
                }

    best_by_label[label]=best

search=pd.DataFrame(rows)
search.to_csv(RUN_DIR/"v27_train_only_ensemble_search.csv",index=False)

locked=pd.DataFrame([best_by_label[l]["rec"] for l in LABELS])
locked.to_csv(RUN_DIR/"v27_locked_config.csv",index=False)

print("LOCKED TRAIN-ONLY CONFIG")
display(locked.round(5))


In [ ]:

# 11. Generate locked validation/test probabilities
def blend_external(j,rec,bh,mu,ch):
    wb=float(rec["w_banglahate"])
    wm=float(rec["w_abusive_muril"])
    wc=float(rec["w_char"])

    if rec["mode"]=="prob":
        return wb*bh[:,j]+wm*mu[:,j]+wc*ch[:,j]

    return sigmoid(
        wb*logit(bh[:,j])
        +wm*logit(mu[:,j])
        +wc*logit(ch[:,j])
    )

v27_val_prob=np.zeros((len(valid),len(LABELS)),float)
v27_test_prob=np.zeros((len(test),len(LABELS)),float)
v27_val_pred=np.zeros((len(valid),len(LABELS)),np.int8)
v27_test_pred=np.zeros((len(test),len(LABELS)),np.int8)

for j,label in enumerate(LABELS):
    rec=best_by_label[label]["rec"]
    t=float(rec["median_threshold"])

    v27_val_prob[:,j]=blend_external(
        j,rec,bh_val,mu_val,char_val
    )
    v27_test_prob[:,j]=blend_external(
        j,rec,bh_test,mu_test,char_test
    )

    v27_val_pred[:,j]=(v27_val_prob[:,j]>=t).astype(np.int8)
    v27_test_pred[:,j]=(v27_test_prob[:,j]>=t).astype(np.int8)

print("✅ Locked V27 predictions generated.")


In [ ]:

# 12. Reconstruct exact V14 validation/test prediction family
li={l:i for i,l in enumerate(LABELS)}

base_v={}
base_t={}

base_v["is_toxic"]=(
    .875*v2v["is_toxic_prob"].to_numpy(float)
    +.125*old_xl_v[1:3,:,0].mean(0)
)
base_t["is_toxic"]=(
    .875*v2t["is_toxic_prob"].to_numpy(float)
    +.125*old_xl_t[1:3,:,0].mean(0)
)

j=li["implicit_toxicity"]
base_v["implicit_toxicity"]=sigmoid(
    .9*logit(v2v["implicit_toxicity_prob"].to_numpy(float))
    +.1*logit(old_bang_v.mean(0)[:,j])
)
base_t["implicit_toxicity"]=sigmoid(
    .9*logit(v2t["implicit_toxicity_prob"].to_numpy(float))
    +.1*logit(old_bang_t.mean(0)[:,j])
)

base_v["sarcasm"]=v2v["sarcasm_prob"].to_numpy(float)
base_t["sarcasm"]=v2t["sarcasm_prob"].to_numpy(float)

j=li["contains_profanity"]
base_v["contains_profanity"]=(
    .95*v2v["contains_profanity_prob"].to_numpy(float)
    +.05*old_bang_v[0,:,j]
)
base_t["contains_profanity"]=(
    .95*v2t["contains_profanity_prob"].to_numpy(float)
    +.05*old_bang_t[0,:,j]
)

j=li["threatening"]
base_v["threatening"]=(
    .95*v2v["threatening_prob"].to_numpy(float)
    +.05*old_xl_v[0,:,j]
)
base_t["threatening"]=(
    .95*v2t["threatening_prob"].to_numpy(float)
    +.05*old_xl_t[0,:,j]
)

j=li["identity_attack"]
base_v["identity_attack"]=(
    .70*v2v["identity_attack_prob"].to_numpy(float)
    +.30*old_bang_v[2,:,j]
)
base_t["identity_attack"]=(
    .70*v2t["identity_attack_prob"].to_numpy(float)
    +.30*old_bang_t[2,:,j]
)

base_v["sexual_content"]=v2v["sexual_content_prob"].to_numpy(float)
base_t["sexual_content"]=v2t["sexual_content_prob"].to_numpy(float)

v14_thresholds={
    "is_toxic":0.6850110311061144,
    "implicit_toxicity":0.1416,
    "sarcasm":0.175,
    "contains_profanity":0.245,
    "threatening":0.16558668018085876,
    "identity_attack":0.3814,
    "sexual_content":0.245,
}

v14_val_pred=np.column_stack([
    (base_v[l]>=v14_thresholds[l]).astype(np.int8)
    for l in LABELS
])
v14_test_pred=np.column_stack([
    (base_t[l]>=v14_thresholds[l]).astype(np.int8)
    for l in LABELS
])

# Test must reconstruct champion exactly.
for j,l in enumerate(LABELS):
    assert np.array_equal(
        v14_test_pred[:,j],
        champ[l].to_numpy(np.int8)
    )

print("✅ Exact V14 reconstructed.")


In [ ]:

# 13. ONE-SHOT untouched validation comparison
y_val=valid[LABELS].to_numpy(np.int8)

def official_from_pred(y,pred):
    scores=[]
    for j,l in enumerate(LABELS):
        scores.append(metric(l,y[:,j],pred[:,j]))
    final=0.5*scores[0]+0.5*np.mean(scores[1:])
    return float(final),scores

v14_score,v14_label_scores=official_from_pred(y_val,v14_val_pred)
v27_score,v27_label_scores=official_from_pred(y_val,v27_val_pred)

report=pd.DataFrame({
    "label":LABELS,
    "v14_validation":v14_label_scores,
    "v27_validation":v27_label_scores,
    "gain":np.asarray(v27_label_scores)-np.asarray(v14_label_scores),
    "true_rate":y_val.mean(0),
    "v14_pred_rate":v14_val_pred.mean(0),
    "v27_pred_rate":v27_val_pred.mean(0),
})

report.to_csv(RUN_DIR/"v27_one_shot_validation.csv",index=False)

print("V14 untouched-validation equivalent:",v14_score)
print("V27 untouched validation:",v27_score)
print("V27 gain:",v27_score-v14_score)
display(report.round(5))


In [ ]:

# 14. Build a conservative label-wise hybrid only from clearly improved labels
# This is the ONLY use of official validation for model selection in V27.

hybrid_val=v14_val_pred.copy()
hybrid_test=v14_test_pred.copy()

selected=[]
for j,label in enumerate(LABELS):
    gain=v27_label_scores[j]-v14_label_scores[j]

    # Stronger margin for rare attributes.
    required=0.010 if label=="is_toxic" else 0.020

    # Avoid catastrophic prevalence shifts vs proven V14 test behavior.
    old_rate=v14_test_pred[:,j].mean()
    new_rate=v27_test_pred[:,j].mean()

    if old_rate==0:
        distribution_ok=True
    else:
        ratio=new_rate/old_rate
        distribution_ok=(0.55 <= ratio <= 1.65)

    if gain>=required and distribution_ok:
        hybrid_val[:,j]=v27_val_pred[:,j]
        hybrid_test[:,j]=v27_test_pred[:,j]
        selected.append(label)

hybrid_score,hybrid_label_scores=official_from_pred(y_val,hybrid_val)
hybrid_gain=hybrid_score-v14_score

# Hard target gate:
# We want enough offline margin to plausibly close the 0.014 public-LB gap.
SUBMIT_CANDIDATE=(
    hybrid_gain >= 0.015
    and len(selected)>=1
)

summary={
    "protected_public_lb":0.586,
    "target_public_lb":0.600,
    "v14_validation":float(v14_score),
    "v27_locked_validation":float(v27_score),
    "hybrid_validation":float(hybrid_score),
    "hybrid_gain_vs_v14":float(hybrid_gain),
    "selected_v27_labels":selected,
    "SUBMIT_CANDIDATE":bool(SUBMIT_CANDIDATE),
}

print(json.dumps(summary,indent=2))

out=champ.copy()
for j,l in enumerate(LABELS):
    out[l]=hybrid_test[:,j]

assert out["ID"].tolist()==test["ID"].tolist()
assert out[LABELS].isin([0,1]).all().all()

out.to_csv(
    RUN_DIR/"submission_v27_target600.csv",
    index=False
)

pure=champ.copy()
for j,l in enumerate(LABELS):
    pure[l]=v27_test_pred[:,j]
pure.to_csv(
    RUN_DIR/"submission_v27_pure_locked.csv",
    index=False
)

comparison=[]
for j,l in enumerate(LABELS):
    comparison.append({
        "label":l,
        "selected_for_hybrid":l in selected,
        "v14_test_positives":int(v14_test_pred[:,j].sum()),
        "v27_test_positives":int(v27_test_pred[:,j].sum()),
        "hybrid_test_positives":int(hybrid_test[:,j].sum()),
        "v27_changed_vs_v14":int((v27_test_pred[:,j]!=v14_test_pred[:,j]).sum()),
        "hybrid_changed_vs_v14":int((hybrid_test[:,j]!=v14_test_pred[:,j]).sum()),
    })

pd.DataFrame(comparison).to_csv(
    RUN_DIR/"v27_test_diagnostics.csv",
    index=False
)

(RUN_DIR/"v27_summary.json").write_text(
    json.dumps(summary,indent=2)
)

if SUBMIT_CANDIDATE:
    print("\\n✅ V27 PASSED TARGET-600 OFFLINE GATE.")
    print("Send v27_results.zip to ChatGPT before Kaggle submission.")
else:
    print("\\n❌ V27 DID NOT PASS THE TARGET-600 OFFLINE GATE.")
    print("Do NOT submit V27. Send v27_results.zip to ChatGPT for diagnosis.")


In [ ]:

# 15. Package all results
np.savez_compressed(
    RUN_DIR/"v27_probabilities.npz",
    banglahate_oof=bh_oof,
    banglahate_validation=bh_val,
    banglahate_test=bh_test,
    abusive_muril_oof=mu_oof,
    abusive_muril_validation=mu_val,
    abusive_muril_test=mu_test,
    char_oof=char_oof,
    char_validation=char_val,
    char_test=char_test,
    v27_validation=v27_val_prob,
    v27_test=v27_test_prob,
)

zip_path=Path("/content/v27_results.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
